# 05 — Vector Database (FAISS)

In [1]:
import sys
sys.path.insert(0, "..")
from src import config, embeddings as emb_mod, vector_store as vs_mod

vectors, metadata, payload = emb_mod.load_embeddings()
print(f"Loaded {vectors.shape[0]} embeddings (dim={vectors.shape[1]})")

backend = vs_mod.resolve_backend()
print("Vector store backend in use:", backend)


Loaded 583 embeddings (dim=768)
Vector store backend in use: faiss


## Build and validate the index

In [2]:
store = vs_mod.VectorStore(dim=vectors.shape[1], backend=backend)
store.build(vectors, metadata)
store.validate()
print("Index built with", len(store.metadata), "vectors.")


2026-09-01 22:32:25,586 | INFO     | src.vector_store | Built faiss-backed vector store with 583 vectors (dim=768)


Index built with 583 vectors.


## Data-integrity check

`number of vectors == number of metadata records`, or raise a clear error.

In [3]:
n_vectors = store._faiss_index.ntotal if store.backend == "faiss" else store._numpy_matrix.shape[0]
assert n_vectors == len(store.metadata), (
    f"MISMATCH: {n_vectors} vectors vs {len(store.metadata)} metadata records"
)
print("Integrity check passed:", n_vectors, "==", len(store.metadata))


Integrity check passed: 583 == 583


## Persist the index

In [4]:
store.save()
print("Saved index to:", config.FAISS_INDEX_PATH)
print("Saved metadata to:", config.FAISS_METADATA_PATH)


2026-09-01 22:32:26,241 | INFO     | src.vector_store | Saved faiss-backed index (583 vectors) to E:\Job Base Programe\ResearchMind\vector_db\faiss_index\index.faiss


Saved index to: E:\Job Base Programe\ResearchMind\vector_db\faiss_index\index.faiss
Saved metadata to: E:\Job Base Programe\ResearchMind\vector_db\faiss_index\metadata.json


## Reload sanity check

In [5]:
reloaded = vs_mod.VectorStore.load()
reloaded.validate()
print("Reload OK —", len(reloaded.metadata), "vectors available for search.")


2026-09-01 22:32:26,884 | INFO     | src.vector_store | Loaded faiss-backed index with 583 vectors from E:\Job Base Programe\ResearchMind\vector_db\faiss_index\index.faiss


Reload OK — 583 vectors available for search.
